# Stage 20A: top-PF-aligned residual screen

Stage 19Cは代理base上のCVでは改善したものの、6.589 baseへ追加すると6.958へ悪化しました。このNotebookは、target-safe public OOF、A130 likelihood PF、SP45 projection、最終60/40 blendを固定200 cutsで再構成し、弱い3係数補正をcross-fitします。提出は作りません。CPUランタイムを使用してください。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess
REPOSITORY_URL='https://github.com/Okada-N13/rogii.git'
repo_dir=Path('/content/ROGII'); drive_root=Path('/content/drive/MyDrive/kaggle/rogii')
artifact_dir=drive_root/'artifacts'; data_dir=drive_root/'data'
if not (repo_dir/'.git').is_dir():
    subprocess.run(['git','clone',REPOSITORY_URL,str(repo_dir)],check=True)
else:
    subprocess.run(['git','-C',str(repo_dir),'pull','--ff-only','origin','main'],check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash','-lc','curl -LsSf https://astral.sh/uv/install.sh | sh'],check=True)
os.environ['PATH']='/root/.local/bin:'+os.environ['PATH']
subprocess.run(['uv','sync','--frozen'],cwd=repo_dir,check=True)
assert (data_dir/'train').is_dir(),data_dir
def run_checked(command):
    result=subprocess.run(command,cwd=repo_dir,text=True,capture_output=True)
    if result.stdout: print(result.stdout,flush=True)
    if result.returncode:
        print(result.stderr,flush=True); raise RuntimeError(f'command failed: {command}')


## 固定artifact

Stage 16B v003 manifestとStage 17Aのwell-isolated public OOFを使います。Stage 19CのLB結果やtest予測は教師・特徴へ使用しません。


In [ ]:
stage16b_run=artifact_dir/'stage16b_testlike_validation_full_v003'
stage17a_run=artifact_dir/'stage17_public_replay_full_v002'
assert (stage16b_run/'pseudo_test_manifest.parquet').is_file(),stage16b_run
assert (stage17a_run/'replay_predictions.parquet').is_file(),stage17a_run
print(stage16b_run,stage17a_run,sep='\n')


## 200-cut A130 replayとcross-fit

5 folds × 5 prefix fractions × 8 cutsをSHA-256で固定抽出します。初回はNumba compileを含みます。途中で25 cutsごとに進捗が表示されます。


In [ ]:
RUN_ID='stage20a_top_pf_alignment_full_v001'; run_dir=artifact_dir/RUN_ID
if not (run_dir/'summary.json').is_file():
    run_checked(['uv','run','rogii-trajectory-base-alignment','--config','configs/experiment/stage20a_top_pf_alignment.yaml','--stage16b-run',str(stage16b_run),'--stage17a-run',str(stage17a_run),'--data-dir',str(data_dir),'--artifact-dir',str(artifact_dir),'--run-id',RUN_ID])
summary=json.loads((run_dir/'summary.json').read_text())
{
 'stage20a_complete':summary['stage20a_complete'], 'promoted_to_stage20b':summary['promoted_to_stage20b'], 'sample_cuts':summary['sample_cuts'],'sample_wells':summary['sample_wells'], 'base_comparison':summary['base_comparison'], 'standard_base_rmse':summary['standard_report']['base_rmse'], 'standard_candidate_rmse':summary['standard_report']['candidate_rmse'], 'standard_delta':summary['standard_report']['rmse_delta'], 'bootstrap_95pct':summary['bootstrap_95pct'],'gates':summary['gates'], 'next_step':summary['next_step'],
}


In [ ]:
import pandas as pd
pd.read_parquet(run_dir/'weight_report.parquet').sort_values('weight')

最後の辞書とweight表を共有してください。Stage 20Aで固定weight 0.10が全gateを通過した場合だけ全eligible cutsと短prefix proxyへ拡張します。ここから直接Kaggleへ提出しません。
